# EFERT | Network Intelligence V3
### Google Colab analytical dashboard | Inline Matplotlib graphics

**Illustrative model built from public information and synthetic data. Not company data.**

Run **Runtime → Run all**. No uploads, Streamlit, tunnels or external map tiles. The optimisation results are embedded Phase 4 outputs; the notebook does not rerun the MILP.

### V3 reliability upgrades
- Primary map is a **saved and displayed Matplotlib PNG**, not a JavaScript visual.
- All six sections use consistently sized inline figures and compact, readable tables.
- Modelled routes are straight connections between approximate coordinates, **not verified driving roads**.
- Inbound and outbound flows are aggregated from actual monthly Phase 4 path-flow records.
- Automated checks include a nonempty PNG, plotted route count and a full notebook execution test.

In [ ]:
import sys, subprocess, importlib.util
required={'pandas':'pandas','numpy':'numpy','matplotlib':'matplotlib'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','-q','install',*missing])
import io, zipfile, base64, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from IPython.display import display, HTML, Markdown, Image
try:
    from ipywidgets import interact, Dropdown
except ImportError:
    class Dropdown:
        def __init__(self,options,value=None,description=''): self.value=value if value is not None else list(options)[0]
    def interact(func,**controls):return func(**{k:v.value for k,v in controls.items()})
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':15,'axes.labelsize':11,'figure.dpi':110,'savefig.dpi':150})
print('Dependencies ready; graphics use inline Matplotlib (no browser JavaScript).')

In [ ]:
# 1. Restore embedded Phase 3 and Phase 4 datasets in the Colab session.
EMBEDDED_DATA = '...'
ROOT=Path('/content/efert_colab_data') if Path('/content').exists() else Path.cwd()/'efert_colab_data'
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_DATA))) as archive:
    archive.extractall(ROOT)
input_names=['nodes','demand_monthly','dc_costs','inventory_params','supply','distances','freight_rates','service','rules','scenarios']
output_names=['annual_optimal_path_flows','annual_optimal_leg_flows','monthly_cost_breakdown','annual_scenarios','five_assumption_sensitivity','switching_thresholds','numerical_audit','independent_subset_validation']
d={n:pd.read_csv(ROOT/'data'/f'{n}.csv') for n in input_names}
d.update({n:pd.read_csv(ROOT/'results'/f'{n}.csv') for n in output_names})
s=json.loads((ROOT/'results'/'summary.json').read_text())
BANNER='Illustrative model built from public information and synthetic data. Not company data.'
GREEN='#12674d'; TEAL='#2a8b76'; GOLD='#e6a33e'; RED='#b94c4c'; INK='#203c35'
COLORS=[GREEN,TEAL,'#80b85a',GOLD,'#507ca0',RED]
from html import escape
GREEN='#12674d'; TEAL='#2a8b76'; GOLD='#e6a33e'; RED='#b94c4c'; INK='#203c35'
COLORS=[GREEN,TEAL,'#80b85a',GOLD,'#507ca0',RED]
BANNER='Illustrative model built from public information and synthetic data. Not company data.'
def heading(name,desc=''):
    display(HTML(f"<div style='max-width:1000px;margin:12px 0 18px;border-top:4px solid {GREEN};padding:12px 0;font:15px Arial'><div style='font-size:12px;color:{GREEN};letter-spacing:1px;font-weight:bold'>EFERT / NETWORK INTELLIGENCE</div><h2 style='font-size:25px;color:{INK};margin:7px 0'>{escape(name)}</h2><p style='color:#52665d;margin:0 0 12px'>{escape(desc)}</p><div style='background:#fff1d7;padding:10px 13px;border-left:4px solid {GOLD};font-size:13px'>{BANNER}</div></div>"))
def kpis(items):
    cards=''.join(f"<div style='flex:1;min-width:165px;border:1px solid #d7e5df;border-radius:8px;padding:14px;background:white'><div style='font-size:12px;color:#5a7066'>{escape(str(k))}</div><div style='font-size:23px;font-weight:800;color:{GREEN};margin:8px 0'>{escape(str(v))}</div><div style='font-size:11px;color:#63776d'>{escape(str(note))}</div></div>" for k,v,note in items)
    display(HTML("<div style='max-width:1000px;display:flex;gap:9px;flex-wrap:wrap;margin:8px 0 16px;font:Arial'>"+cards+'</div>'))
def table(df,formats=None,rows=12):
    z=df.head(rows).copy()
    if formats:
        for col,fmt in formats.items():
            if col in z:z[col]=z[col].map(lambda v:fmt.format(v) if pd.notna(v) else '—')
    display(HTML("<div style='max-width:1000px;overflow-x:auto;margin:8px 0 14px'>"+z.to_html(index=False,border=0,classes='efert-tbl')+"</div><style>.efert-tbl{border-collapse:collapse;font:12px Arial;width:100%;color:#243e34}.efert-tbl th{background:#e7f1eb;text-align:left;padding:8px}.efert-tbl td{padding:7px;border-bottom:1px solid #e8eeeb}.efert-tbl tr:nth-child(even){background:#f8faf8}</style>"))
def insight(text):
    display(HTML(f"<div style='max-width:970px;padding:12px 14px;margin:10px 0 18px;border-left:4px solid {GREEN};background:#edf6f0;font:14px/1.5 Arial;color:{INK}'><b>Management interpretation:</b> {escape(text)}</div>"))
def finish(fig,name=None):
    fig.tight_layout(pad=1.5)
    if name:fig.savefig(ROOT/name,bbox_inches='tight',facecolor='white',dpi=150)
    plt.show();plt.close(fig)
print(f'Loaded {len(input_names)} input tables, {len(output_names)} result tables and solver summary.')

## 01 | Executive command centre
The baseline is a hypothetical network with all six candidate DCs leased. It is not EFERT’s observed distribution system.

**Management interpretation:** The modelled saving is small relative to annual costs (approximately 0.37%). The decision is conditional on synthetic demand, assumed freight and DC throughput capacity, and the illustrative six-DC baseline.

In [ ]:
heading('01 / Executive command centre','Annual optimisation decision and cost performance, 2025.')
kpis([('Annual optimised cost',f"PKR {s['optimal_PKR']/1e9:.3f}bn",'[DERIVED] annual MILP'),('Illustrative savings',f"PKR {s['savings_PKR']/1e6:.2f}m",f"{s['savings_pct']:.3f}% versus hypothetical baseline"),('DCs selected',f"{len(s['open_DCs'])}/6",', '.join(s['open_DCs'])),('Regional demand',f"{s['annual_tonnes']/1e6:.3f}m t",'[SYNTHETIC] 2025')])
m=d['monthly_cost_breakdown'].copy();m['cost_m']=m['total_PKR']/1e6
fig,ax=plt.subplots(figsize=(10,3.4));ax.plot(m['month'],m['cost_m'],color=GREEN,marker='o',linewidth=2);ax.fill_between(range(len(m)),m['cost_m'],alpha=.13,color=GREEN)
ax.set_title('Monthly optimised logistics cost',loc='left',fontweight='bold');ax.set_ylabel('PKR million');ax.tick_params(axis='x',rotation=35);ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2);finish(fig)
comp=m[['freight_PKR','handling_PKR','fixed_PKR','inventory_PKR']].sum();comp.index=['Freight','Handling','Fixed DC','Inventory reserve']
fig,ax=plt.subplots(figsize=(10,2.6));ax.barh(comp.index,comp.values/1e9,color=COLORS[:4]);ax.invert_yaxis();ax.set_xlabel('PKR billion');ax.set_title('Annual cost composition',loc='left',fontweight='bold');ax.spines[['top','right']].set_visible(False);ax.grid(axis='x',alpha=.15);finish(fig)
insight(f"Five candidate DCs are selected. Savings of {s['savings_pct']:.3f}% are modest and depend on synthetic demand and assumed freight costs. The six-DC baseline is hypothetical, not EFERT's observed network.")

## 02 | Geographic network flows
Choose a month, DC and route type. Line width represents modelled tonnes. **Lines are straight connections between approximate city coordinates, not verified road routes.**

**Map note:** The map now uses longitude/latitude axes and local Plotly scatter traces, so it does not depend on external geographic projection assets. It shows modelled straight-line connections, not verified roads.

In [ ]:
heading('02 / Network intelligence','Reliable inline Matplotlib route map with modelled flows.')
nodes=d['nodes'].set_index('node_id');paths=d['annual_optimal_path_flows'].copy()
assert {'month','plant','dc','zone','tonnes'}.issubset(paths.columns)
MAP_PNG=ROOT/'EFERT_Network_Map_V3.png'
MAP_STATS={}
def network_map(month,dc='All DCs',route='All'):
    p=paths[paths['month'].astype(str).eq(str(month))].copy()
    if dc!='All DCs':p=p[p['dc'].eq(dc)]
    inbound=p.groupby(['plant','dc'],as_index=False)['tonnes'].sum().rename(columns={'plant':'a','dc':'b'})
    outbound=p.groupby(['dc','zone'],as_index=False)['tonnes'].sum().rename(columns={'dc':'a','zone':'b'})
    inbound['leg']='Inbound';outbound['leg']='Outbound'
    pairs=pd.concat([inbound,outbound],ignore_index=True)
    if route!='All':pairs=pairs[pairs['leg'].eq(route)]
    pairs=pairs[pairs['tonnes']>0].copy()
    fig,ax=plt.subplots(figsize=(10,7.0),facecolor='white')
    maxflow=max(pairs['tonnes'].max() if len(pairs) else 1,1)
    drawn=0
    for r in pairs.itertuples(index=False):
        if r.a not in nodes.index or r.b not in nodes.index:continue
        a,b=nodes.loc[r.a],nodes.loc[r.b]
        # Both plants are co-located: a slight horizontal offset distinguishes their labels/markers.
        xa=float(a['lon'])+(-.12 if r.a=='P01' else .12 if r.a=='P02' else 0)
        xb=float(b['lon'])+(-.12 if r.b=='P01' else .12 if r.b=='P02' else 0)
        ax.plot([xa,xb],[float(a['lat']),float(b['lat'])],color=TEAL if r.leg=='Inbound' else GOLD,
                alpha=.35 if r.leg=='Inbound' else .48,linewidth=.8+4.5*float(r.tonnes)/maxflow,zorder=1)
        drawn+=1
    active=set(s['open_DCs'])
    for typ,marker,color,size in [('zone','o',GOLD,44),('dc','s',GREEN,110),('plant','D',RED,100)]:
        frame=nodes[nodes['type'].eq(typ)]
        for nid,r in frame.iterrows():
            x=float(r['lon'])+(-.12 if nid=='P01' else .12 if nid=='P02' else 0);y=float(r['lat'])
            enabled=(typ!='dc' or nid in active)
            ax.scatter(x,y,marker=marker,s=size,facecolor=color if enabled else 'white',edgecolor=INK if enabled else '#8e9b95',linewidth=1.1,zorder=5)
            if typ=='plant':label=f'{nid} Daharki'
            elif typ=='dc':label=f'{nid} {r["city"]}'+('' if enabled else ' (closed)')
            else:label=str(r['city'])
            offset=(5,8) if typ=='plant' else (6,-11 if typ=='zone' else 7)
            ax.annotate(label,(x,y),xytext=offset,textcoords='offset points',fontsize=7.5 if typ=='zone' else 8.5,color=INK,zorder=6)
    ax.set_xlim(66.3,75.1);ax.set_ylim(23.7,32.5);ax.set_aspect('equal',adjustable='box')
    ax.set_xlabel('Longitude (°E)');ax.set_ylabel('Latitude (°N)')
    ax.set_title(f'Modelled distribution routes | {month} | {dc} | {route}',loc='left',fontweight='bold',pad=15)
    ax.grid(alpha=.15);ax.spines[['top','right']].set_visible(False)
    handles=[Line2D([0],[0],color=TEAL,lw=3,label='Plant → DC'),Line2D([0],[0],color=GOLD,lw=3,label='DC → zone'),
             Line2D([0],[0],marker='D',color='w',markerfacecolor=RED,markeredgecolor=INK,markersize=9,label='Plant'),
             Line2D([0],[0],marker='s',color='w',markerfacecolor=GREEN,markeredgecolor=INK,markersize=9,label='Active DC'),
             Line2D([0],[0],marker='o',color='w',markerfacecolor=GOLD,markeredgecolor=INK,markersize=8,label='Demand zone')]
    ax.legend(handles=handles,loc='lower left',fontsize=9,frameon=True,ncol=2)
    finish(fig,'EFERT_Network_Map_V3.png')
    assert MAP_PNG.is_file() and MAP_PNG.stat().st_size>20000,'Network PNG is missing or empty'
    MAP_STATS.update({'month':month,'route_connections':drawn,'bytes':MAP_PNG.stat().st_size,'tonnes':float(outbound['tonnes'].sum())})
    util=outbound.groupby('a',as_index=False)['tonnes'].sum().rename(columns={'a':'dc'})
    util=d['dc_costs'][['candidate_id','city','capacity']].merge(util,left_on='candidate_id',right_on='dc',how='left')
    util['tonnes']=util['tonnes'].fillna(0);util['utilisation_pct']=100*util['tonnes']/util['capacity'];util['dc']=util['candidate_id']
    table(util[['dc','city','tonnes','capacity','utilisation_pct']],{'tonnes':'{:,.0f}','capacity':'{:,.0f}','utilisation_pct':'{:.1f}%'},rows=6)
    insight(f"{drawn} route connections drawn; selected DC outbound flow {outbound['tonnes'].sum():,.0f} tonnes. Width scales with modelled flow. Coordinates and straight lines are illustrative, not verified road distances.")
months=sorted(paths['month'].astype(str).unique())
# Render the first map unconditionally, so it is visible even if Colab widgets fail.
network_map(months[0])
print('Optional: rerun the network_map function with another month, DC or leg. Example: network_map("2025-06", "All DCs", "Outbound")')

## 03 | Cost and inventory analytics
Inventory is a **fixed per-open-DC safety-stock holding reserve**, not a dynamic inventory-flow or warehouse storage-capacity model.

In [ ]:
heading('03 / Cost and inventory analytics','Transparent cost reconciliation and provisional inventory assumptions.')
kpis([('Annual freight',f"PKR {m['freight_PKR'].sum()/1e9:.3f}bn",'[DERIVED]'),('Inventory reserve',f"PKR {m['inventory_PKR'].sum()/1e6:.2f}m",'[ASSUMED] holding basis'),('Cost per tonne',f"PKR {s['optimal_PKR']/s['annual_tonnes']:,.0f}",'Annual objective / tonnes')])
fig,ax=plt.subplots(figsize=(10,3.8));bottom=np.zeros(len(m))
for j,col in enumerate(['freight_PKR','handling_PKR','fixed_PKR','inventory_PKR']):
    values=m[col].to_numpy()/1e6;ax.bar(m['month'],values,bottom=bottom,color=COLORS[j],label=col.replace('_PKR','').title());bottom+=values
ax.set_title('Monthly logistics cost composition',loc='left',fontweight='bold');ax.set_ylabel('PKR million');ax.tick_params(axis='x',rotation=35);ax.legend(loc='upper center',bbox_to_anchor=(.5,-.23),ncol=4,frameon=False);ax.spines[['top','right']].set_visible(False);finish(fig)
inv=d['inventory_params'].copy();table(inv[['parameter','value','unit','tag','verification']],rows=10)
print(f'Inventory register contains {len(inv)} parameters; first 10 shown. The inventory model is a fixed per-open-DC reserve, not a dynamic stock-flow model.')
m['component_sum']=m[['freight_PKR','handling_PKR','fixed_PKR','inventory_PKR']].sum(axis=1);m['difference_PKR']=m['total_PKR']-m['component_sum']
table(m[['month','tonnes','total_PKR','difference_PKR']],{'tonnes':'{:,.0f}','total_PKR':'{:,.0f}','difference_PKR':'{:.5f}'},rows=12)
insight('Freight dominates the modelled objective. Inventory reserve costs use assumed holding parameters and must not be interpreted as observed EFERT inventory expense.')

## 04 | Six scenario stress tests
These are **precomputed annual MILP results**. Infeasible scenarios are excluded from cost bars and reported separately.

**Management interpretation:** Infeasibility is a capacity warning, not a cost of zero. A contingency plan would require additional supply or reduced demand commitments.

In [ ]:
heading('04 / Scenario stress tests','Six shocks re-optimised in Phase 4. Infeasible scenarios have no assigned cost.')
sc=d['annual_scenarios'].copy();sc['change_pct']=(sc['cost_PKR']/s['optimal_PKR']-1)*100
sc['annual_cost_bn']=sc['cost_PKR']/1e9
sc['annual_cost_label']=sc['annual_cost_bn'].map(lambda v:f'PKR {v:,.3f}bn' if pd.notna(v) else 'INFEASIBLE')
sc['change_label']=sc['change_pct'].map(lambda v:f'{v:+.2f}%' if pd.notna(v) else '—')
table(sc[['scenario_id','scenario','status','annual_cost_label','change_label','selected_DCs']],rows=7)
f=sc[sc['status'].eq('OPTIMAL')].copy()
fig,ax=plt.subplots(figsize=(10,3.6));bars=ax.bar(f['scenario'],f['annual_cost_bn'],color=COLORS[:len(f)])
for b,v in zip(bars,f['annual_cost_bn']):ax.text(b.get_x()+b.get_width()/2,v+.08,f'{v:.2f}',ha='center',fontsize=9)
ax.set_ylim(0,f['annual_cost_bn'].max()*1.15);ax.set_title('Annual cost under feasible scenarios',loc='left',fontweight='bold');ax.set_ylabel('PKR billion');ax.tick_params(axis='x',rotation=15);ax.spines[['top','right']].set_visible(False);finish(fig)
for row in sc[sc['status'].ne('OPTIMAL')].itertuples():
    reason=('Demand exceeds modelled production capacity.' if row.scenario_id=='S02' else 'Remaining plant capacity cannot meet demand.' if row.scenario_id=='S04' else 'Model constraints cannot all be met.')
    print(f'INFEASIBLE | {row.scenario}: {reason}')
insight('The largest feasible cost increase is associated with the fuel shock. Infeasible shocks require additional capacity or revised assumptions; zero-cost bars would be misleading.')

## 05 | Five-assumption sensitivity and switching thresholds
All 25 sensitivity records come from Phase 4 re-optimisation. Blank/infeasible cells are **not** treated as stable networks.

In [ ]:
heading('05 / Sensitivity and decisions','Twenty-five re-optimisations across five assumptions and switching thresholds.')
sen=d['five_assumption_sensitivity'].copy();sen['assumption_change_pct']=(sen['multiplier']-1)*100;sen['annual_cost_bn']=sen['annual_cost_PKR']/1e9
fig,axes=plt.subplots(2,3,figsize=(10,5.6),sharex=True)
for ax,parameter in zip(axes.flat,sorted(sen['parameter'].unique())):
    z=sen[sen['parameter'].eq(parameter)].sort_values('multiplier');good=z[z['status'].eq('OPTIMAL')]
    ax.plot(good['assumption_change_pct'],good['annual_cost_bn'],marker='o',color=GREEN,lw=2)
    bad=z[z['status'].ne('OPTIMAL')]
    if len(bad):ax.scatter(bad['assumption_change_pct'],[s['optimal_PKR']/1e9]*len(bad),marker='x',color=RED,s=55,label='Infeasible')
    ax.set_title(parameter.replace('_',' ').title(),fontsize=11);ax.grid(alpha=.2);ax.spines[['top','right']].set_visible(False)
for ax in axes.flat[len(sen['parameter'].unique()):]:ax.axis('off')
for ax in axes[-1]:ax.set_xlabel('Assumption change (%)')
axes[0,0].set_ylabel('Annual cost (PKR bn)');axes[1,0].set_ylabel('Annual cost (PKR bn)')
fig.suptitle('Sensitivity of annual objective to five assumptions',fontsize=15,fontweight='bold');finish(fig)
heat=sen.pivot(index='parameter',columns='multiplier',values='network_flip').copy()
fig,ax=plt.subplots(figsize=(9,2.7));values=np.full(heat.shape,np.nan)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        z=sen[(sen['parameter']==heat.index[i])&(sen['multiplier']==heat.columns[j])]
        if len(z) and z.iloc[0]['status']=='OPTIMAL':values[i,j]=int(str(z.iloc[0]['network_flip']).lower()=='true')
from matplotlib.colors import ListedColormap
ax.imshow(np.ma.masked_invalid(values),cmap=ListedColormap(['#e3f0e7',GOLD]),vmin=0,vmax=1,aspect='auto')
ax.set_xticks(range(len(heat.columns)),[f'{x:.0%}' for x in heat.columns]);ax.set_yticks(range(len(heat.index)),[x.replace('_',' ') for x in heat.index]);ax.set_title('Network selection change: green = stable, amber = changed, white = infeasible',loc='left',fontsize=11)
for i in range(values.shape[0]):
    for j in range(values.shape[1]):ax.text(j,i,'—' if np.isnan(values[i,j]) else 'FLIP' if values[i,j] else 'OK',ha='center',va='center',fontsize=9)
finish(fig)
thr=d['switching_thresholds'].copy();thr['percent_of_base']=thr['multiplier']*100
table(thr[['parameter','threshold_type','percent_of_base','interpretation','tag']],{'percent_of_base':'{:.2f}%'},rows=8)
insight('Demand and DC throughput capacity are the most decision-sensitive tested assumptions. Thresholds are approximate and conditional on all other inputs remaining unchanged.')

## 06 | Evidence, provenance and independent audit
The provenance chart counts tagged **rows**, not the percentage of factual claims that have been externally verified.

**Audit interpretation:** 488 recorded checks and an independent LP enumeration support mathematical consistency. They do not establish that the inputs represent EFERT’s actual operations.

In [ ]:
heading('06 / Evidence and model audit','Tagged input provenance, independent verification and outstanding evidence.')
rows=[]
for name in input_names:
    frame=d[name]
    if 'tag' in frame.columns:
        for tag,count in frame['tag'].fillna('UNLABELLED').value_counts().items():rows.append({'dataset':name,'tag':tag,'rows':int(count)})
prov=pd.DataFrame(rows);totals=prov.groupby('tag',as_index=False)['rows'].sum()
fig,ax=plt.subplots(figsize=(7,3.2));ax.barh(totals['tag'],totals['rows'],color=COLORS[:len(totals)])
for i,v in enumerate(totals['rows']):ax.text(v+4,i,f'{v:,}',va='center',fontsize=10)
ax.set_xlim(0,totals['rows'].max()*1.20);ax.set_title('Actual tagged input rows (not verified-data percentage)',loc='left',fontweight='bold');ax.set_xlabel('Number of rows');ax.spines[['top','right']].set_visible(False);finish(fig)
table(prov.sort_values(['dataset','tag']),{'rows':'{:,.0f}'},rows=15)
audit=d['numerical_audit'].copy();passed=audit['passed'].astype(str).str.lower().eq('true');subset=d['independent_subset_validation']
kpis([('Recorded checks',f'{passed.sum()}/{len(audit)}','Baseline and optimal'),('Feasible DC subsets',str(len(subset)),'Of 63 enumerated'),('Independent objective gap',f"PKR {s['independent_gap_PKR']:.6f}",'Fixed-open LP comparison')])
print('Unresolved evidence and methodological limitations:')
for issue in s['limitations']:print(' •',issue)
print(' • Road distances, freight quotes, exact coordinates and instructor acceptance of synthetic inventory remain unverified.')
table(subset,rows=7)
insight('The recorded numerical checks verify internal consistency of the illustrative model. They do not establish the real-world accuracy of EFERT routes, costs, inventory or savings.')

## 07 | Automated verification
Checks include numerical reconciliation, input provenance, **nonempty Matplotlib PNG**, raster content, modelled route count and demand reconciliation. These do not verify real road geometry.

In [ ]:
# Assertion-based integrity tests, executed on the embedded project records.
checks={}
def check(name,condition):
    checks[name]=bool(condition)
    if not condition:raise AssertionError(name)
check('All 10 Phase 3 input CSVs',len(input_names)==10 and all(len(d[n])>0 for n in input_names))
check('All 8 Phase 4 output CSVs',len(output_names)==8 and all(len(d[n])>0 for n in output_names))
check('Annual cost savings reconcile',np.isclose(s['baseline_PKR']-s['optimal_PKR'],s['savings_PKR'],atol=.01))
check('Monthly objective sums to annual optimum',np.isclose(m['total_PKR'].sum(),s['optimal_PKR'],atol=.01))
check('Monthly cost components reconcile',np.allclose(m['component_sum'],m['total_PKR'],atol=.01))
check('Annual demand sums to summary',np.isclose(m['tonnes'].sum(),s['annual_tonnes'],atol=.01))
check('Six stress scenarios plus base',len(sc)==7 and set(sc['scenario_id'])=={'S00','S01','S02','S03','S04','S05','S06'})
check('Infeasible scenarios have no numeric costs',sc.loc[sc['status'].ne('OPTIMAL'),'cost_PKR'].isna().all())
check('25 sensitivity runs',len(sen)==25 and sen['parameter'].nunique()==5)
check('Recorded audit checks pass',passed.all() and len(audit)==488)
check('Independent objective agrees with MILP',abs(s['independent_optimum_PKR']-s['optimal_PKR'])<.01)
check('Selected DCs match independent enumeration',s['open_DCs']==s['independent_open_DCs'])
check('Input provenance is based on rows',prov['rows'].sum()==sum(len(d[n]) for n in input_names if 'tag' in d[n]))
print(f'✓ {sum(checks.values())}/{len(checks)} notebook integrity tests passed.')
display(pd.DataFrame([{'test':k,'passed':v} for k,v in checks.items()]))
# V3 map-specific validation: a saved raster must exist, be sizeable and contain actual ink.
from PIL import Image as PILImage
check('Matplotlib map PNG exists and exceeds 20KB',MAP_PNG.exists() and MAP_PNG.stat().st_size>20000)
with PILImage.open(MAP_PNG) as raster:
    arr=np.asarray(raster.convert('RGB'))
    colored=np.any(arr<225,axis=2)
    check('Map raster has meaningful nonwhite content',arr.shape[0]>=600 and arr.shape[1]>=800 and colored.mean()>.008)
check('Map draws actual positive-flow connections',MAP_STATS.get('route_connections',0)>=5 and MAP_STATS.get('tonnes',0)>0)
check('Map flow matches January demand',np.isclose(MAP_STATS['tonnes'],float(m.loc[m['month'].eq(months[0]),'tonnes'].iloc[0]),atol=.05))
print(f'V3 map checks passed; {MAP_STATS["route_connections"]} connections, {MAP_STATS["bytes"]:,} PNG bytes.')
print(f'TOTAL: {sum(checks.values())}/{len(checks)} checks passed.')

In [ ]:
# Optional: export dashboard-ready data and a verification report from Colab.
export=ROOT/'EFERT_Colab_Verification_Report.csv'
pd.DataFrame([{'test':k,'passed':v} for k,v in checks.items()]).to_csv(export,index=False)
print('Report saved to:',export)
print('Data files available under:',ROOT)
# In Google Colab, uncomment the next two lines to download the report:
# from google.colab import files
# files.download(str(export))
print('Map file:',MAP_PNG)